In [ ]:
from itertools import combinations
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns
from IPython.display import display

# Statistics

In this tutorial, we continue working with the heart disease dataset to build a solid foundation in exploratory statistics. We start by characterizing the data using descriptive statistics. Next, we examine the properties of the normal distribution and apply statistical tests to assess whether variables in the dataset are normally distributed. Finally, we conduct correlation analyses and critically discuss why correlation does not imply causation.

In [ ]:
cleveland = pd.read_csv(
    "./heart_disease_data.csv", 
    delimiter=",", 
    header=0,
    na_values="?",
    index_col=0
)
cleveland.columns = pd.Index(
    [
        "age", # ... in years
        "sex", # 1 = male, 0 = female
        "chestpain", # type of chest pain (4 categories)
        "restbp", # resting blood pressure at admission (mmHg)
        "serumchol", # serum cholesterol (mg/dl)
        "fastbloodsugar", # fasting blood sugar >120mg/dl (1 = yes, 0 = no)
        "restecg", # resting ECG results (0 = normal, 1 = ST-T wave abnormal, 2 = probable or definite ventricular hypertrophy)
        "thalach", # maximum heart rate
        "exang", # exercise induced angina (1 = yes, 0 = no)
        "oldpeak", # ST depression induced by exercise (relative to resting state!)
        "slope", # slope of peak exercise ST segment (1 = up, 2 = flat, 3 = down)
        "ca", # number of major vessels coloured by fluorospy (range from 0 to 3)
        "thal", # 3 = normal, 6 = fixed defect, 7 = reversable defect
        "num", # diagnosis: 0 = <50% diameter narrowing, 1 = >50% diameter narrowing
    ]
)
# description of diagnosis restricted to absence of heart disease (num=0) or presence of heart disease (num={1, 2, 3, 4})
cleveland["num"] = cleveland["num"].replace({2: 1, 3: 1, 4: 1}) 
cleveland.head(5)

In [ ]:
# Restrict the analysis to complete cases
n_before = len(cleveland)
cleveland = cleveland.dropna().copy()
n_removed = n_before - len(cleveland)

print(f"Removed {n_removed} rows with missing data.")
print(f"Remaining complete cases: {len(cleveland)}")

In [ ]:
cleveland["chestpain"] = pd.Categorical(cleveland["chestpain"])
cleveland["chestpain"] = cleveland["chestpain"].cat.rename_categories(
    {1: "typical angina", 2: "atypical angina", 3: "non-anginal pain", 4: "asymptomatic"}
)

# 1 = male, 0 = female
cleveland["sex"] = pd.Categorical(cleveland["sex"])
cleveland["sex"] = cleveland["sex"].cat.rename_categories(
    {1: "male", 0: "female"}
)
# fasting blood sugar >120mg/dl (1 = yes, 0 = no)
cleveland["fastbloodsugar"] = pd.Categorical(cleveland["fastbloodsugar"])
cleveland["fastbloodsugar"] = cleveland["fastbloodsugar"].cat.rename_categories(
    {1: ">120mg/dl", 0: "<120mg/dl"}
)
# resting ECG results (0 = normal, 1 = ST-T wave abnormal, 2 = probable or definite ventricular hypertrophy)
cleveland["restecg"] = pd.Categorical(cleveland["restecg"])
cleveland["restecg"] = cleveland["restecg"].cat.rename_categories(
    {0: "normal", 1: "ST-T abnormal", 2: "ventricular hypertrophy"}
)
# exercise induced angina (1 = yes, 0 = no)
cleveland["exang"] = pd.Categorical(cleveland["exang"])
cleveland["exang"] = cleveland["exang"].cat.rename_categories(
    {0: "no", 1: "yes"}
)
# slope of peak exercise ST segment (1 = up, 2 = flat, 3 = down)
cleveland["slope"] = pd.Categorical(cleveland["slope"])
cleveland["slope"] = cleveland["slope"].cat.rename_categories(
    {1: "up", 2: "flat", 3: "down"}
)
# 3 = normal, 6 = fixed defect, 7 = reversable defect
cleveland["thal"] = pd.Categorical(cleveland["thal"])
cleveland["thal"] = cleveland["thal"].cat.rename_categories(
    {3: "normal", 6: "fixed defect", 7: "reversable defect"}
)
# diagnosis: 0 = <50% diameter narrowing, 1 = >50% diameter narrowing
cleveland["num"] = pd.Categorical(cleveland["num"])
cleveland["num"] = cleveland["num"].cat.rename_categories(
    {0: "<50%", 1: ">50%"}
)
cleveland.head(5)

In [ ]:
cleveland["ca"] = cleveland["ca"].astype("int32")

## Descriptive Statistics

Descriptive statistics help us summarize and understand the main characteristics of a dataset. This tutorial focuses on descriptive statistics, which summarize the observed sample; inferential statistics come later when we draw conclusions about a broader population from sample data.

In this section, we will look at four main categories:

1. **Measures of frequency**
2. **Measures of central tendency**
3. **Measures of dispersion**
4. **Measures of position**

> **Important note:**  
> This dataset is a clinical dataset, not a population-based public health registry.  
> Therefore, we can compute **counts, proportions, and ratios within the dataset**, but we should be careful not to interpret them as population incidence or prevalence rates.


### Measures of frequency

Measures of frequency describe how often categories or events occur.

In epidemiology, common examples include:
- ratios
- proportions
- rates
- incidence
- prevalence
- mortality rates

In this dataset, we do **not** have enough information to compute incidence or prevalence in the general population.  
However, we **can** describe frequency in the sample using:

- **counts**
- **proportions**
- **cross-tabulations**
- **ratios**

In [ ]:
# Frequency tables for selected categorical variables
for col in ["sex", "chestpain", "num", "slope"]:
    print(f"\n=== {col} ===")
    print("Counts:")
    print(cleveland[col].value_counts(dropna=False))
    print("\nProportions:")
    print(cleveland[col].value_counts(normalize=True, dropna=False).round(3))

Proportions can be calculated accross all values, columns or rows.

In [ ]:
# Cross-tabulation: diagnosis by sex
diag_by_sex = pd.crosstab(cleveland["sex"], cleveland["num"], margins=True)
diag_by_sex_prop = pd.crosstab(cleveland["sex"], cleveland["num"], normalize="index").round(3)

print("Counts:")
display(diag_by_sex)

print("Row proportions:")
display(diag_by_sex_prop)

In [ ]:
# Example of a simple ratio within the dataset:
# ratio of males to females
sex_counts = cleveland["sex"].value_counts()

if "male" in sex_counts.index and "female" in sex_counts.index:
    male_female_ratio = sex_counts["male"] / sex_counts["female"]
    print(f"Male-to-female ratio in this dataset: {male_female_ratio:.2f}")
else:
    print("Male/female labels not found exactly as expected.")

### Measures of central tendency

Measures of central tendency summarize the “center” or typical value of a distribution.

Common measures are:
- **mean**
- **median**
- **mode**

We will illustrate these using the numeric variables:
- `age`
- `restbp`
- `serumchol`
- `thalach`
- `oldpeak`
- `ca`

In [ ]:
numeric_vars = ["age", "restbp", "serumchol", "thalach", "oldpeak", "ca"]

central_tendency = pd.DataFrame({
    "mean": cleveland[numeric_vars].mean(),
    "median": cleveland[numeric_vars].median(),
    "mode": cleveland[numeric_vars].mode().iloc[0] # we will keep only the first mode, but a dataset/variable can have multiple modes.
}).round(2)

central_tendency

In [ ]:
# Example: visualize mean and median on the age distribution
plt.figure(figsize=(7, 4))
sns.histplot(data=cleveland, x="age", bins=15, kde=True)

plt.axvline(cleveland["age"].mean(), color="red", linestyle="--", linewidth=1.5, label="mean")
plt.axvline(cleveland["age"].median(), color="black", linestyle="--", linewidth=1.5, label="median")

plt.title("Age distribution with mean and median")
plt.xlabel("Age (years)")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

### Measures of dispersion

Measures of dispersion describe how spread out the data are.

They help us understand whether a variable is:
- relatively homogeneous (tightly clustered)
- or heterogeneous (widely spread)

We distinguish between:

#### Absolute measures of dispersion
These have the **same unit** as the original variable:
- range
- variance
- standard deviation
- interquartile range (IQR)
- quartile deviation
- mean absolute deviation

#### Relative measures of dispersion
These are **unit-free** and useful for comparing variables with different scales:
- coefficient of range
- coefficient of variation
- coefficient of standard deviation
- coefficient of quartile deviation

In [ ]:
def descriptive_dispersion(series):
    s = series.dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    mean_val = s.mean()
    min_val, max_val = s.min(), s.max()
    iqr_val = q3 - q1

    return pd.Series({
        "min": min_val,
        "max": max_val,
        "range": max_val - min_val,
        "variance": s.var(ddof=1),
        "std_dev": s.std(ddof=1),
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr_val,
        "quartile_deviation": iqr_val / 2,
        "mean_absolute_deviation": (s - mean_val).abs().mean(),
        "coef_range": (max_val - min_val) / (max_val + min_val) if (max_val + min_val) != 0 else np.nan,
        "coef_variation": s.std(ddof=1) / mean_val if mean_val != 0 else np.nan,
        "coef_quartile_deviation": iqr_val / (q3 + q1) if (q3 + q1) != 0 else np.nan
    })

dispersion_table = cleveland[numeric_vars].apply(descriptive_dispersion).T.round(3)
dispersion_table

In [ ]:
# Visual comparison of dispersion for two variables
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=cleveland, y="restbp", ax=axs[0])
axs[0].set_title("Dispersion of resting blood pressure")
axs[0].set_ylabel("restbp (mmHg)")

sns.boxplot(data=cleveland, y="serumchol", ax=axs[1])
axs[1].set_title("Dispersion of serum cholesterol")
axs[1].set_ylabel("serumchol (mg/dl)")

plt.tight_layout()
plt.show()

### Measures of position

Measures of position describe where a value lies relative to the rest of the dataset.

Common examples:
- **quartiles** → divide the data into 4 equal parts
- **deciles** → divide the data into 10 equal parts
- **percentiles** → divide the data into 100 equal parts

These are especially useful because they are less sensitive to a few extreme observations than the mean.

In [ ]:
position_var = "age"

quartiles = cleveland[position_var].quantile([0.25, 0.50, 0.75])
deciles = cleveland[position_var].quantile(np.arange(0.1, 1.0, 0.1))

print(f"Quartiles for {position_var}:")
display(quartiles.to_frame(name=position_var))

print(f"Deciles for {position_var}:")
display(deciles.to_frame(name=position_var))

In [ ]:
# Show the quartiles on a histogram
q1 = cleveland[position_var].quantile(0.25)
q2 = cleveland[position_var].quantile(0.50)
q3 = cleveland[position_var].quantile(0.75)

plt.figure(figsize=(7, 4))
sns.histplot(data=cleveland, x=position_var, bins=15, kde=True)

plt.axvline(q1, color="black", linestyle="--", linewidth=1, label="Q1")
plt.axvline(q2, color="red", linestyle="--", linewidth=1, label="Median (Q2)")
plt.axvline(q3, color="black", linestyle="--", linewidth=1, label="Q3")

plt.title(f"Measures of position for {position_var}")
plt.xlabel("Age (years)")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

## Exercise 1: describe another variable

So far we described `age`. Now do the same for `serumchol` (serum cholesterol).

Task:
1. Compute the **mean** and the **median** of `serumchol` and print them.
2. Compute the **quartiles** (25%, 50%, 75%) of `serumchol`.

Hint: look at the code cells in *Measures of central tendency* and *Measures of position* above. You only need to change the variable name.

In [ ]:
# 1. Mean and median of serumchol
# YOUR CODE HERE

# 2. Quartiles of serumchol
# YOUR CODE HERE

### Normality vs non-normality

A normal distribution is symmetric and bell-shaped.

To assess normality, we can use:
- visual inspection of histograms
- Q–Q plots
- Shapiro–Wilk test

> The Shapiro–Wilk test checks the null hypothesis that the data come from a normal distribution.  
> A small p-value suggests evidence against normality.

In [ ]:
# Compare several distributions visually
fig, axs = plt.subplots(2, 3, figsize=(14, 8))
axs = axs.flatten()

for ax, col in zip(axs, numeric_vars):
    sns.histplot(data=cleveland, x=col, bins=15, kde=True, ax=ax)
    ax.set_title(col)

plt.tight_layout()
plt.show()

In [ ]:
# Q-Q plots for all numeric variables
n_vars = len(numeric_vars)
n_cols = 3
n_rows = int(np.ceil(n_vars / n_cols))

fig, axs = plt.subplots(n_rows, n_cols, figsize=(14, 4.5 * n_rows))
axs = axs.flatten()

for ax, col in zip(axs, numeric_vars):
    stats.probplot(cleveland[col].dropna(), dist="norm", plot=ax)
    ax.set_title(f"Q-Q plot for {col}")

# Hide unused subplots
for ax in axs[n_vars:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
normality_results = []

for col in numeric_vars:
    stat, p = stats.shapiro(cleveland[col].dropna())
    normality_results.append({
        "variable": col,
        "W_statistic": stat,
        "p_value": p
    })

normality_table = pd.DataFrame(normality_results).round(4)
normality_table

## Correlation analysis

Correlation quantifies the strength and direction of an association between two variables. It does **not** automatically imply a causal relationship.

Before computing a coefficient, always inspect a scatter plot.

In [ ]:
pplot = sns.pairplot(
    data=cleveland,
    x_vars=numeric_vars, y_vars=numeric_vars,
)

Let's quantify the correlation between these pairs of variables.

(Check the documentation of [Python's `itertools`](https://docs.python.org/3/library/itertools.html) to understand better how to easily obtain pairs of variables (with or without replacement) for example.)

In [ ]:
for comb in combinations(numeric_vars, 2):
    print(f" {comb[0]} - {comb[1]}")
    print(f"\tSpearman Rank Correlation {stats.spearmanr(cleveland[comb[0]], cleveland[comb[1]]).statistic: .3f}, p-value {stats.spearmanr(cleveland[comb[0]], cleveland[comb[1]]).pvalue: .3f}")
    print(f"\tPearson Correlation  {stats.pearsonr(cleveland[comb[0]], cleveland[comb[1]]).statistic: .3f}, p-value {stats.pearsonr(cleveland[comb[0]], cleveland[comb[1]]).pvalue: .3f}")

## Exercise 2: correlation between two variables

Task:
1. Compute the **Spearman correlation** between `age` and `thalach` (maximum heart rate) and print the correlation coefficient and the p-value.
2. Is the correlation positive or negative? What does this mean in words?

Hint: stats.spearmanr(cleveland["x"], cleveland["y"]) returns the correlation (.statistic) and the p-value (.pvalue).

In [ ]:
# 1. Spearman correlation between age and thalach
# YOUR CODE HERE

# 2. Positive or negative? What does it mean?
# YOUR ANSWER HERE